# K_max Convergence Analysis: Does Our Firm-Similarity Measure Stabilize?

**Purpose**: Determine whether the firm-pair Bhattacharyya Coefficient rankings — our proxy for technological similarity in the M&A prediction task — stabilize as we relax the maximum number of mixture components K_max ∈ {10, 15, 20, 25, 30}.

**Why this matters**: The Week 2 design phase found that BC rankings are moderately stable in bulk (Spearman ρ ≈ 0.80) but the top tail — the firm pairs we would flag as M&A candidates — changes materially with K_max (top-50 overlap as low as 22%). Before locking in a production specification, we need to know whether this is a tunable detail or a fundamental property of the methodology.

**The framing**: This is a *methodology resolution run*, not a hyperparameter tuning exercise. The point is to replace an arbitrary default with an empirically justified one — or, if convergence does not emerge, to formally classify the similarity signal as sensitivity-conditioned rather than pretending it is fixed.

**Pre-registered decision rule** (locked before sweep launch):

| Outcome | Production K_max | Week 3 reporting |
|---|---|---|
| Converged (ρ > 0.95 AND top-50 > 80% from K* onward) | K* = smallest converged K_max | Single primary specification + neighbor as robustness check |
| Not converged | Operational default (largest K_max tested) | All top-pair conclusions classified as "robust" or "model-sensitive" |

**Prerequisites**:
- Sweep results in `output/kmax_sweep/` (run `scripts/run_kmax_sweep.py` on AWS)
- Patent vector inputs in `output/week2_inputs/`

**Reading guide**: Markdown cells tell the story. Code cells produce evidence. A reader interested only in findings can skim the markdown; a reader interested in the methodology can read the code.

## Section 1: Setup and Data Summary

We fitted Bayesian Gaussian Mixture Models for ~7,949 firms at five levels of maximum technological granularity (K_max = 10, 15, 20, 25, 30). Each firm's patent portfolio is represented as a probability distribution — a weighted mixture of Gaussian components in 50-dimensional UMAP embedding space. Each component represents a distinct "technology area" the firm operates in.

Before we look at convergence, it is worth understanding the two-tier structure of our sample:

- **Single-Gaussian tier** (5–49 patents): ~6,304 firms. These have K=1 by construction — a single Gaussian summarizes the portfolio. Their representations are *invariant* to K_max. They form the stable bedrock of the pairwise similarity matrix.
- **GMM tier** (50+ patents): ~1,645 firms holding ~93% of all patents. These are where the granularity question matters. Any K_max sensitivity in the rankings is concentrated here.

This separation is the most important architectural fact about the analysis. **Whatever instability we find is concentrated in the firms that hold most of the patents** — which is also where the rich distributional information lives.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports — reuse functions from the sweep script for consistency
import sys
sys.path.insert(0, str(Path("..").resolve()))
from scripts.run_kmax_sweep import (
    load_gmm_results,
    bc_mixture,
    bc_component_matrix,
    compute_effective_k_summary,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

SWEEP_DIR = Path("../output/kmax_sweep")
INPUTS_DIR = Path("../output/week2_inputs")


In [ ]:
# Deep-dive firm identifications (from patent title pattern matching, see Section 5)
DEEP_DIVE_FIRMS = {
    "006066": "IBM",
    "012141": "Intel",
    "024800": "Qualcomm",
    "160329": "Google/Alphabet",
    "020779": "Cisco Systems",
}

# Load convergence summary (the ground truth for the verdict)
with open(SWEEP_DIR / "convergence_summary.json") as f:
    summary = json.load(f)

K_MAX_VALUES = summary["k_max_values"]
print(f"K_max values tested: {K_MAX_VALUES}")
print(f"Convergence verdict: {summary['convergence_verdict']}")
print(f"Converged at K_max: {summary.get('converged_at_kmax')}")
print(f"Total runtime: {summary['timing']['total_hours']} hours")


In [ ]:
# Load all GMM results (one list per K_max)
all_results = {}
for k_max in K_MAX_VALUES:
    path = SWEEP_DIR / f"firm_gmm_parameters_k{k_max}.parquet"
    all_results[k_max] = load_gmm_results(str(path))
    print(f"K_max={k_max}: {len(all_results[k_max])} firms loaded")

# Load all BC matrices
bc_matrices = {}
for k_max in K_MAX_VALUES:
    path = SWEEP_DIR / f"bc_matrix_all_k{k_max}.npz"
    data = np.load(path, allow_pickle=True)
    bc_matrices[k_max] = (data["gvkeys"].tolist(), data["bc_matrix"])
    print(f"K_max={k_max}: BC matrix {bc_matrices[k_max][1].shape}")

# Load excluded firms log
excluded = pd.read_csv(SWEEP_DIR / "excluded_firms.csv")
print(f"Excluded firms: {len(excluded):,}")


In [ ]:
# Tier breakdown
results_first = all_results[K_MAX_VALUES[0]]  # Tier classifications same across K_max
n_gmm = sum(1 for r in results_first if r["tier"] == "gmm")
n_sg = sum(1 for r in results_first if r["tier"] == "single_gaussian")
n_total = n_gmm + n_sg

n_gmm_patents = sum(r["n_patents"] for r in results_first if r["tier"] == "gmm")
n_sg_patents = sum(r["n_patents"] for r in results_first if r["tier"] == "single_gaussian")
n_excluded_patents = excluded["n_patents"].sum() if "n_patents" in excluded.columns else 0
n_total_patents = n_gmm_patents + n_sg_patents + n_excluded_patents

# Pairs: triangle without diagonal
n_pairs_total = n_total * (n_total - 1) // 2
n_pairs_gmm = n_gmm * (n_gmm - 1) // 2

summary_table = pd.DataFrame({
    "Metric": [
        "Total firms in BC analysis",
        "  ... GMM tier (50+ patents)",
        "  ... Single-Gaussian tier (5-49 patents)",
        "Excluded firms (<5 patents)",
        "Patents in GMM tier",
        "Patents in single-Gaussian tier",
        "Patents excluded",
        "Pairs evaluated per K_max",
        "  ... involving at least one GMM-tier firm",
        "K_max values tested",
    ],
    "Value": [
        f"{n_total:,}",
        f"{n_gmm:,} ({n_gmm/n_total:.1%})",
        f"{n_sg:,} ({n_sg/n_total:.1%})",
        f"{len(excluded):,}",
        f"{n_gmm_patents:,} ({n_gmm_patents/n_total_patents:.1%})",
        f"{n_sg_patents:,} ({n_sg_patents/n_total_patents:.1%})",
        f"{n_excluded_patents:,} ({n_excluded_patents/n_total_patents:.1%})",
        f"{n_pairs_total:,}",
        f"{n_pairs_total - (n_sg * (n_sg-1) // 2):,}",
        ", ".join(str(k) for k in K_MAX_VALUES),
    ],
})
print(summary_table.to_string(index=False))


## Section 2: How Many Technology Areas Do Firms Have?

The first question is whether firms *use* the granularity we give them. If we allow up to 30 technology areas and most firms still settle on 10, the model is not constrained — granularity is irrelevant. If most firms hit the ceiling at every K_max, the model is asking for more structure than we're allowing.

The Bayesian prior we use (a Dirichlet Process with concentration γ=1.0) implies an expected number of components that scales logarithmically with firm size: E[K] ≈ γ · log(n). For a firm with 1,000 patents, this predicts E[K] ≈ 7. For 10,000 patents, E[K] ≈ 11. This is the economic prior we expect — larger firms operate in more technology areas, but with diminishing returns.

We measure two things:
1. **The progression of effective K** (mean, P25, P75, P90) as K_max increases
2. **The ceiling-hit rate** — what fraction of firms have effective K equal to K_max, indicating they wanted more components than allowed

### Visualization 2A: Effective K progression vs K_max

Each line shows a percentile of the effective K distribution across the 1,645 GMM-tier firms. The dashed diagonal is the K_max ceiling — points on the diagonal mean firms are saturating the limit. The gap between P90 and the ceiling tells us whether the most diversified firms have stopped asking for more components.

In [ ]:
# TODO: Compute effective K stats per K_max for GMM-tier firms only
# For each K_max, extract list of n_components from gmm-tier results
# Compute mean, p25, p50, p75, p90, ceiling rate

k_progression = []
for k_max in K_MAX_VALUES:
    gmm_results = [r for r in all_results[k_max] if r["tier"] == "gmm"]
    ks = np.array([r["n_components"] for r in gmm_results])
    ceiling_rate = np.mean(ks >= k_max - 0.5) * 100  # at or near ceiling
    k_progression.append({
        "k_max": k_max,
        "mean": ks.mean(),
        "p25": np.percentile(ks, 25),
        "p50": np.percentile(ks, 50),
        "p75": np.percentile(ks, 75),
        "p90": np.percentile(ks, 90),
        "ceiling_rate_pct": ceiling_rate,
    })
k_df = pd.DataFrame(k_progression)
print(k_df.to_string(index=False))

# Plot: mean with P25-P75 band, P90 line, ceiling diagonal
fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(k_df["k_max"], k_df["p25"], k_df["p75"], alpha=0.3, color="steelblue", label="P25-P75 band")
ax.plot(k_df["k_max"], k_df["mean"], "o-", color="steelblue", linewidth=2, label="Mean effective K")
ax.plot(k_df["k_max"], k_df["p90"], "s--", color="coral", linewidth=2, label="P90 effective K")
ax.plot(k_df["k_max"], k_df["k_max"], "k:", alpha=0.5, label="K_max ceiling")
ax.set_xlabel("K_max (allowed maximum)")
ax.set_ylabel("Effective K (after Bayesian pruning)")
ax.set_title("Do firms saturate as we allow more technology areas?")
ax.legend()

# Annotate ceiling rate as text
for _, row in k_df.iterrows():
    ax.annotate(f'{row["ceiling_rate_pct"]:.0f}%', (row["k_max"], row["p90"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9, color="coral")

plt.tight_layout()
plt.savefig("03_viz2a_k_progression.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 2B: Effective K distribution at each K_max (violin plot)

The violin plots show the full distribution of effective K across the 1,645 GMM-tier firms. If the violins shift upward and stretch as K_max grows, firms are using the additional granularity. If they stay roughly fixed in shape, the model is data-driven and the K_max parameter is non-binding for the typical firm.

In [ ]:
# Build long-form dataframe for violin plot
violin_rows = []
for k_max in K_MAX_VALUES:
    gmm_results = [r for r in all_results[k_max] if r["tier"] == "gmm"]
    for r in gmm_results:
        violin_rows.append({"K_max": k_max, "Effective K": r["n_components"]})
violin_df = pd.DataFrame(violin_rows)

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=violin_df, x="K_max", y="Effective K", inner="quartile",
               palette="Blues", ax=ax)
ax.set_title("Distribution of effective K across 1,645 GMM-tier firms")
ax.set_xlabel("K_max setting")
ax.set_ylabel("Effective K (post-pruning)")
plt.tight_layout()
plt.savefig("03_viz2b_k_violin.png", dpi=120, bbox_inches="tight")
plt.show()


### Sanity check: Does K scale with firm size?

The Dirichlet Process prior predicts E[K] ≈ γ · log(n). We check whether the model behaves consistently with this expectation by computing the rank correlation between firm patent count and effective K. A strong positive sub-linear relationship is evidence that the GMM is capturing real economic structure rather than mechanically creating one component per N patents.

In [ ]:
from scipy.stats import spearmanr

# For the production K_max (largest tested), check K vs firm size
k_max_largest = K_MAX_VALUES[-1]
gmm_results = [r for r in all_results[k_max_largest] if r["tier"] == "gmm"]
sizes = np.array([r["n_patents"] for r in gmm_results])
ks = np.array([r["n_components"] for r in gmm_results])

rho, _ = spearmanr(sizes, ks)
print(f"Spearman rho (firm size vs effective K) at K_max={k_max_largest}: {rho:.3f}")

# Visualize: scatter with log x-axis and a log-fit line
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sizes, ks, alpha=0.3, s=15, color="steelblue")
# Fit K = a + b * log(n)
log_sizes = np.log(sizes)
b, a = np.polyfit(log_sizes, ks, 1)
x_fit = np.logspace(np.log10(sizes.min()), np.log10(sizes.max()), 100)
ax.plot(x_fit, a + b * np.log(x_fit), "r-", linewidth=2,
        label=f"K = {a:.2f} + {b:.2f}·log(n)")
ax.set_xscale("log")
ax.set_xlabel("Firm patent count (log scale)")
ax.set_ylabel(f"Effective K at K_max={k_max_largest}")
ax.set_title(f"Effective K scales with firm size (Spearman ρ = {rho:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("03_viz2c_k_vs_size.png", dpi=120, bbox_inches="tight")
plt.show()


**[INTERPRETATION — populate after results]**

What we expect to find:
- Mean effective K should grow with K_max but increasingly slowly if convergence is happening
- P90 effective K is the diagnostic for the most diversified firms — if it keeps climbing, granularity is still binding for them
- Ceiling rate should drop as K_max grows; if it stays >20% even at K_max=30, the model is asking for more structure
- Firm-size vs K relationship should be sub-linear (log scaling) — strong evidence of economic interpretability

Findings: `[FILL IN AFTER RESULTS]`

## Section 3: Bulk Ranking Stability

We compute the Bhattacharyya Coefficient (BC) for all ~31.6 million firm pairs at each K_max setting. The BC is a number between 0 and 1 measuring distributional overlap — values near 1 mean two firms have very similar technology distributions, values near 0 mean they are almost disjoint.

The first question is: **across all 31.6M pairs, how much do the rankings change between K_max values?** This is the bulk stability question. The Spearman rank correlation is the natural measure.

**Why bulk stability matters less than you might think**: Most pairs are firms in completely different technology spaces — a biotech firm and a semiconductor firm, a chemicals company and a software company. These pairs have BC near zero regardless of K_max. The bulk Spearman correlation is dominated by these obvious non-matches. If we want to know whether *the candidates we care about* are stable, we need Section 4.

### Visualization 3A: Spearman rho across K_max transitions

The convergence threshold is ρ > 0.95. Adjacent transitions (10→15, 15→20, etc.) are the primary measure. Non-adjacent (10→30, etc.) shows cumulative drift.

In [ ]:
# Pull adjacent and non-adjacent comparison results from convergence_summary.json
adjacent = summary["adjacent_comparisons"]
non_adjacent = summary["non_adjacent_comparisons"]

# Build comparison dataframe
adj_df = pd.DataFrame(adjacent)
nonadj_df = pd.DataFrame(non_adjacent)
print("Adjacent comparisons:")
print(adj_df[["k_max_a", "k_max_b", "spearman_rho", "kendall_tau",
              "top_50_overlap_pct", "top_100_overlap_pct", "top_200_overlap_pct",
              "mean_nn5_overlap_pct"]].to_string(index=False))

# Plot: Spearman rho across adjacent transitions, with non-adjacent as separate points
fig, ax = plt.subplots(figsize=(10, 5))
adj_labels = [f"{r['k_max_a']}→{r['k_max_b']}" for _, r in adj_df.iterrows()]
ax.plot(range(len(adj_df)), adj_df["spearman_rho"], "o-", color="steelblue",
        linewidth=2, markersize=10, label="Adjacent")
# Non-adjacent as separate annotations
for _, r in nonadj_df.iterrows():
    ax.scatter([len(adj_df) + 0.5], [r["spearman_rho"]], color="coral", s=80, alpha=0.7,
               label=f'{r["k_max_a"]}→{r["k_max_b"]}' if _ < 1 else None)
    ax.annotate(f'{r["k_max_a"]}→{r["k_max_b"]}',
                (len(adj_df) + 0.5, r["spearman_rho"]),
                textcoords="offset points", xytext=(8, 0), fontsize=8)

ax.axhline(y=0.95, color="red", linestyle="--", alpha=0.6, label="Convergence threshold (ρ > 0.95)")
ax.set_xticks(range(len(adj_df)))
ax.set_xticklabels(adj_labels)
ax.set_xlabel("K_max transition")
ax.set_ylabel("Spearman ρ (BC ranking correlation)")
ax.set_title("Bulk ranking stability across K_max transitions")
ax.set_ylim([min(0.7, adj_df["spearman_rho"].min() - 0.05), 1.01])
ax.legend()
plt.tight_layout()
plt.savefig("03_viz3a_spearman.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 3B: BC scatter (adjacent K_max settings)

A 2D hexbin scatter where each point is a firm pair: x-axis is BC at K_max=N, y-axis is BC at K_max=N+5. If rankings are stable, points fall on the diagonal. The interesting feature: the bulk lies tight along the diagonal at low BC values, but the upper-right corner (high-BC pairs — the ones that matter for M&A prediction) shows more scatter.

In [ ]:
# Pick two adjacent K_max settings near the convergence point (or just two interesting ones)
k_a, k_b = K_MAX_VALUES[1], K_MAX_VALUES[2]  # e.g., 15 vs 20
gvkeys_a, bc_a = bc_matrices[k_a]
gvkeys_b, bc_b = bc_matrices[k_b]

# Both matrices have the same firm ordering (sorted gvkey from sweep script)
assert gvkeys_a == gvkeys_b

# Extract upper-triangle (unique pairs)
n = len(gvkeys_a)
iu = np.triu_indices(n, k=1)
flat_a = bc_a[iu]
flat_b = bc_b[iu]

# Hexbin
fig, ax = plt.subplots(figsize=(8, 8))
hb = ax.hexbin(flat_a, flat_b, gridsize=80, cmap="viridis", bins="log", mincnt=1)
ax.plot([0, 1], [0, 1], "r-", linewidth=1, alpha=0.7, label="y = x")
ax.set_xlabel(f"BC at K_max={k_a}")
ax.set_ylabel(f"BC at K_max={k_b}")
ax.set_title(f"Pairwise BC: K_max={k_a} vs K_max={k_b}\nBulk on diagonal, top tail more scattered")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
fig.colorbar(hb, ax=ax, label="Pair count (log scale)")
ax.legend()
plt.tight_layout()
plt.savefig(f"03_viz3b_bc_scatter_k{k_a}_vs_k{k_b}.png", dpi=120, bbox_inches="tight")
plt.show()


**[INTERPRETATION — populate after results]**

The bulk Spearman correlation is the "good news" measure. For the vast majority of firm pairs — the 99%+ that are not close technology competitors — the K_max parameter is essentially irrelevant. Whether we define technology areas at K_max=15 or K_max=25, firm A and firm B that operate in completely different technology spaces remain clearly dissimilar.

But this is not the question we care about. The question is whether the *highest-BC pairs* — the ones we would flag as M&A candidates — are stable. That is Section 4.

Findings: `[FILL IN AFTER RESULTS]`

## Section 4: Top-Tail Instability — The Candidates We Care About

The whole point of the BC measure is to identify firm pairs with high technological overlap as candidates for M&A. So the question that matters is: **among the top-50 (or top-100, or top-200) most similar pairs, how many of the same pairs survive across K_max settings?**

This is *not* a question that bulk Spearman ρ answers. A correlation of 0.95 across 31.6 million pairs is consistent with the top-50 being completely shuffled. Top-tail and bulk are orthogonal measures of stability.

### Visualization 4A: K_max × K_max top-100 overlap heatmap

This is the central diagnostic for the candidate set. Each cell shows what fraction of the top-100 most similar pairs at one K_max are also in the top-100 at another K_max. Diagonal is 100% by construction. Off-diagonal cells tell us how much the candidate set churns. A "block" of high overlap from K_max=N onward indicates a stable region.

In [ ]:
def top_k_pair_set(bc_matrix: np.ndarray, k: int) -> set:
    """Return the set of (i, j) indices for the top-k pairs by BC value."""
    n = bc_matrix.shape[0]
    iu = np.triu_indices(n, k=1)
    flat = bc_matrix[iu]
    top_idx = np.argsort(-flat)[:k]
    return set(zip(iu[0][top_idx].tolist(), iu[1][top_idx].tolist()))

# Compute K_max × K_max overlap matrix for top-100 pairs
top_k = 100
top_pair_sets = {}
for k_max in K_MAX_VALUES:
    _, bc_mat = bc_matrices[k_max]
    top_pair_sets[k_max] = top_k_pair_set(bc_mat, top_k)

n_kmax = len(K_MAX_VALUES)
overlap_matrix = np.zeros((n_kmax, n_kmax))
for i, ka in enumerate(K_MAX_VALUES):
    for j, kb in enumerate(K_MAX_VALUES):
        overlap = len(top_pair_sets[ka] & top_pair_sets[kb]) / top_k * 100
        overlap_matrix[i, j] = overlap

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(overlap_matrix, annot=True, fmt=".0f", cmap="RdYlGn", vmin=0, vmax=100,
            xticklabels=K_MAX_VALUES, yticklabels=K_MAX_VALUES,
            cbar_kws={"label": "Top-100 pair overlap (%)"}, ax=ax)
ax.set_xlabel("K_max")
ax.set_ylabel("K_max")
ax.set_title(f"Top-{top_k} pair overlap matrix across K_max settings")
plt.tight_layout()
plt.savefig("03_viz4a_top100_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 4B: Rank trajectories of the top-200 pairs

For the top-200 pairs at K_max=10, we trace where each pair ranks at K_max=15, 20, 25, 30. Each line is a firm pair. Pairs that remain in the top-200 at all K_max are colored steelblue (the "robust core"). Pairs that drop out are colored coral (the "volatile periphery"). The y-axis is log-rank, inverted so rank 1 is at the top.

In [ ]:
# Compute rank of each pair at each K_max, starting from top-200 at K_max[0]
N_PAIRS_TRACE = 200
k_first = K_MAX_VALUES[0]
_, bc_first = bc_matrices[k_first]
n = bc_first.shape[0]
iu = np.triu_indices(n, k=1)
flat_first = bc_first[iu]
top_idx_first = np.argsort(-flat_first)[:N_PAIRS_TRACE]
# Convert to (i, j) tuples
top_pairs = list(zip(iu[0][top_idx_first].tolist(), iu[1][top_idx_first].tolist()))

# For each K_max, compute the rank of each of these pairs
rank_trajectories = np.zeros((N_PAIRS_TRACE, len(K_MAX_VALUES)), dtype=np.int64)
for k_idx, k_max in enumerate(K_MAX_VALUES):
    _, bc_mat = bc_matrices[k_max]
    flat = bc_mat[iu]
    # Argsort gives positions; we want the rank of each (i,j) in the original order
    rank_array = np.empty_like(flat, dtype=np.int64)
    rank_array[np.argsort(-flat)] = np.arange(len(flat))
    # Map (i,j) → flat index
    for p_idx, (i_, j_) in enumerate(top_pairs):
        # Find position in iu
        flat_pos = np.where((iu[0] == i_) & (iu[1] == j_))[0][0]
        rank_trajectories[p_idx, k_idx] = rank_array[flat_pos] + 1  # 1-indexed

# Classify: robust = always in top-200, volatile = drops out at some K_max
final_ranks = rank_trajectories[:, -1]
robust_mask = (rank_trajectories <= N_PAIRS_TRACE).all(axis=1)

fig, ax = plt.subplots(figsize=(11, 7))
for p_idx in range(N_PAIRS_TRACE):
    color = "steelblue" if robust_mask[p_idx] else "coral"
    alpha = 0.5 if robust_mask[p_idx] else 0.25
    ax.plot(range(len(K_MAX_VALUES)), rank_trajectories[p_idx], "-",
            color=color, alpha=alpha, linewidth=0.8)

ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xticks(range(len(K_MAX_VALUES)))
ax.set_xticklabels([f"K_max={k}" for k in K_MAX_VALUES])
ax.set_ylabel("Pair rank (log scale, top = best)")
ax.axhline(y=N_PAIRS_TRACE, color="black", linestyle=":", alpha=0.5,
           label=f"Top-{N_PAIRS_TRACE} threshold")
ax.set_title(f"Rank trajectories of top-{N_PAIRS_TRACE} pairs from K_max={k_first}")
robust_count = robust_mask.sum()
ax.text(0.02, 0.98, f"Robust (in top-{N_PAIRS_TRACE} at all K_max): {robust_count}/{N_PAIRS_TRACE}",
        transform=ax.transAxes, fontsize=11, verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("03_viz4b_rank_trajectories.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 4C: Per-firm nearest-neighbor stability

For each firm, we look at its top-5 most similar firms at each K_max setting. How many of those 5 neighbors persist across K_max values? If a firm has 5/5 of its top-5 neighbors stable from K_max=15 onward, the firm has a robust technology profile. If only 2/5 persist, the firm's similarity profile is sensitive to model specification.

The histogram shows the distribution of NN-5 overlap across all firms. The P10 line marks the worst 10% — the firms whose similarity neighborhoods are most fragile.

In [ ]:
# Use the adjacent comparison closest to the converged K_max (or the middle one)
# The convergence_summary already has mean_nn5_overlap_pct per pair, but we need the
# full distribution. We compute it here from BC matrices.

def per_firm_top_k_neighbors(bc_matrix: np.ndarray, k: int = 5) -> list[set]:
    """For each row in bc_matrix, return set of top-k column indices (excluding self)."""
    n = bc_matrix.shape[0]
    neighbor_sets = []
    for i in range(n):
        row = bc_matrix[i].copy()
        row[i] = -1  # exclude self
        top = np.argsort(-row)[:k]
        neighbor_sets.append(set(top.tolist()))
    return neighbor_sets

# Compare two adjacent K_max settings
k_a, k_b = K_MAX_VALUES[1], K_MAX_VALUES[2]  # e.g., 15 vs 20
_, bc_a = bc_matrices[k_a]
_, bc_b = bc_matrices[k_b]
nn_a = per_firm_top_k_neighbors(bc_a, k=5)
nn_b = per_firm_top_k_neighbors(bc_b, k=5)
overlaps = np.array([len(a & b) for a, b in zip(nn_a, nn_b)])  # 0-5

fig, ax = plt.subplots(figsize=(10, 5))
counts = np.bincount(overlaps, minlength=6)
ax.bar(range(6), counts, color="steelblue", edgecolor="black", alpha=0.8)
for i, c in enumerate(counts):
    ax.text(i, c + max(counts) * 0.01, f"{c:,}\n({c/len(overlaps):.1%})",
            ha="center", va="bottom", fontsize=9)
ax.set_xlabel(f"Number of nearest neighbors preserved (out of 5)\nK_max={k_a} → K_max={k_b}")
ax.set_ylabel("Firms")
ax.set_title(f"Per-firm nearest-neighbor stability between K_max={k_a} and K_max={k_b}")
ax.set_xticks(range(6))
plt.tight_layout()
plt.savefig(f"03_viz4c_nn5_distribution_k{k_a}_k{k_b}.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Median NN-5 overlap: {np.median(overlaps):.1f}/5")
print(f"P10 NN-5 overlap: {np.percentile(overlaps, 10):.1f}/5")
print(f"Firms with 5/5 stable: {(overlaps == 5).sum()} ({(overlaps == 5).mean():.1%})")
print(f"Firms with ≤2/5 stable: {(overlaps <= 2).sum()} ({(overlaps <= 2).mean():.1%})")


**[INTERPRETATION — populate after results]**

This is the section that decides the verdict. Key questions:
- If top-100 overlap stabilizes at >80% from some K_max forward — convergence in the actionable region
- If rank trajectories show a stable "robust core" — even without full convergence, that core is what we should report as M&A candidates
- If most firms have ≥4/5 stable neighbors — the per-firm similarity profile is trustworthy

Findings: `[FILL IN AFTER RESULTS]`

## Section 5: Firm Deep-Dives — Making the Abstract Concrete

Statistics tell one story; named examples tell another. We pick five firms spanning the diversification spectrum and trace their GMM representation across K_max values. The selected firms (identified by patent title patterns from the v3 data):

| gvkey | Firm | Patents | Expected behavior |
|---|---|---|---|
| 006066 | **IBM** | ~157K | Mega-conglomerate. Effective K should keep growing — IBM operates across semiconductors, software, AI, quantum, cloud, services. |
| 012141 | **Intel** | ~47K | Focused semiconductor giant. K should plateau by mid-K_max — Intel's portfolio is deep but in a small number of well-defined areas. |
| 024800 | **Qualcomm** | ~36K | Ultra-focused on telecom IP and CDMA. K should stabilize early. |
| 160329 | **Google/Alphabet** | ~30K | Software/internet, mid-diversified. Expected K growth between Intel and IBM. |
| 020779 | **Cisco Systems** | ~18K | Networking equipment specialist. K should stabilize. |

These firms span:
- **Size**: 18K to 157K patents (1 order of magnitude)
- **Diversification**: from single-domain (Cisco) to mega-conglomerate (IBM)
- **Era**: from established (IBM, Cisco) to relatively new (Google/Alphabet)

If the model behaves sensibly, we should see: IBM's K keeps growing, the focused firms plateau, Google sits in between.

### Visualization 5A: Effective K vs K_max for the deep-dive firms

A single line per firm. Firms whose K plateau demonstrate convergence; firms whose K keeps climbing demonstrate that granularity is still binding for them.

In [ ]:
# Build per-firm K progression
firm_k_progression = {gk: [] for gk in DEEP_DIVE_FIRMS}
firm_n_patents = {}

for k_max in K_MAX_VALUES:
    by_gvkey = {r["gvkey"]: r for r in all_results[k_max]}
    for gk in DEEP_DIVE_FIRMS:
        if gk in by_gvkey:
            firm_k_progression[gk].append(by_gvkey[gk]["n_components"])
            firm_n_patents[gk] = by_gvkey[gk]["n_patents"]
        else:
            firm_k_progression[gk].append(np.nan)

# Verify all firms exist
for gk, name in DEEP_DIVE_FIRMS.items():
    n = firm_n_patents.get(gk, "MISSING")
    print(f"  {name} (gvkey={gk}): n_patents={n}, K progression={firm_k_progression[gk]}")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Set1", n_colors=len(DEEP_DIVE_FIRMS))
for (gk, name), color in zip(DEEP_DIVE_FIRMS.items(), colors):
    label = f"{name} (n={firm_n_patents.get(gk, '?'):,})"
    ax.plot(K_MAX_VALUES, firm_k_progression[gk], "o-", color=color,
            linewidth=2, markersize=8, label=label)
ax.plot(K_MAX_VALUES, K_MAX_VALUES, "k:", alpha=0.4, label="Ceiling")
ax.set_xlabel("K_max")
ax.set_ylabel("Effective K")
ax.set_title("Effective K progression for 5 named firms")
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.savefig("03_viz5a_firm_k_progression.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 5B: Component weight evolution per firm

For each deep-dive firm, a stacked area chart showing how the mixing weights of its GMM components change as K_max increases. Each colored band is a component (sorted by weight at K_max=10). At K_max=10, IBM might have weights [0.20, 0.15, 0.12, 0.10, ...]. At K_max=20, some components split — the stacked area shows whether existing components fragment or whether truly new ones appear.

In [ ]:
# For each firm, build a (n_kmax, max_K) matrix of sorted weights
fig, axes = plt.subplots(len(DEEP_DIVE_FIRMS), 1, figsize=(11, 3 * len(DEEP_DIVE_FIRMS)),
                          sharex=True)

for ax, (gk, name) in zip(axes, DEEP_DIVE_FIRMS.items()):
    weight_matrix = []
    max_k = 0
    for k_max in K_MAX_VALUES:
        by_gvkey = {r["gvkey"]: r for r in all_results[k_max]}
        if gk in by_gvkey:
            w = sorted(by_gvkey[gk]["weights"], reverse=True)
            weight_matrix.append(w)
            max_k = max(max_k, len(w))
        else:
            weight_matrix.append([])
    # Pad to max_k
    padded = np.zeros((len(K_MAX_VALUES), max_k))
    for i, w in enumerate(weight_matrix):
        padded[i, :len(w)] = w

    # Stacked area
    bottom = np.zeros(len(K_MAX_VALUES))
    colors_comp = sns.color_palette("husl", n_colors=max_k)
    for k_idx in range(max_k):
        ax.bar(K_MAX_VALUES, padded[:, k_idx], bottom=bottom, width=2.5,
               color=colors_comp[k_idx], edgecolor="white", linewidth=0.3)
        bottom += padded[:, k_idx]
    ax.set_ylabel(f"{name}\n(n={firm_n_patents.get(gk, '?'):,})")
    ax.set_ylim([0, 1.02])
    ax.set_xticks(K_MAX_VALUES)

axes[-1].set_xlabel("K_max")
fig.suptitle("Component weight evolution as K_max increases", y=1.001)
plt.tight_layout()
plt.savefig("03_viz5b_firm_weights.png", dpi=120, bbox_inches="tight")
plt.show()


### Visualization 5C: Top-5 nearest neighbors per firm at each K_max

For each deep-dive firm, what are its 5 most similar firms at each K_max setting? Persistent neighbors (appearing at all K_max) are bolded; transient neighbors are plain. This makes the abstract NN stability statistic concrete: "Intel's most similar firm is X at K_max=10, but Y at K_max=20.

In [ ]:
# For each deep-dive firm, find its top-5 BC neighbors at each K_max
def top_neighbors_for_firm(gvkey: str, bc_matrix: np.ndarray, gvkeys: list, k: int = 5):
    if gvkey not in gvkeys:
        return []
    idx = gvkeys.index(gvkey)
    row = bc_matrix[idx].copy()
    row[idx] = -1
    top_idx = np.argsort(-row)[:k]
    return [(gvkeys[i], float(row[i])) for i in top_idx]

# Build a dataframe per firm: rows = neighbor rank, columns = K_max
neighbor_tables = {}
for gk, name in DEEP_DIVE_FIRMS.items():
    rows = []
    for k_max in K_MAX_VALUES:
        gvkeys, bc_mat = bc_matrices[k_max]
        top = top_neighbors_for_firm(gk, bc_mat, gvkeys, k=5)
        for rank, (n_gvkey, bc_val) in enumerate(top, 1):
            n_name = DEEP_DIVE_FIRMS.get(n_gvkey, n_gvkey)
            rows.append({"K_max": k_max, "Rank": rank, "Neighbor": n_name, "BC": round(bc_val, 4)})
    df = pd.DataFrame(rows)
    neighbor_tables[gk] = df
    pivot = df.pivot(index="Rank", columns="K_max", values="Neighbor")
    print(f"\n=== {name} (gvkey={gk}) — top-5 neighbors by K_max ===")
    print(pivot.to_string())


**[INTERPRETATION — populate after results]**

For each firm, write 2-3 sentences interpreting what K_max sensitivity means in concrete terms.

- **IBM**: `[FILL IN]`
- **Intel**: `[FILL IN]`
- **Qualcomm**: `[FILL IN]`
- **Google/Alphabet**: `[FILL IN]`
- **Cisco**: `[FILL IN]`

## Section 6: The Convergence Verdict

We synthesize Sections 2-5 into the verdict. The pre-registered decision rule (locked before the sweep ran):

> **K\* = the smallest K_max such that ALL subsequent adjacent comparisons from K\* onward pass both:**
> - Spearman ρ > 0.95
> - Top-50 pair overlap > 80%

This is the "persistent stability" rule. A single passing transition followed by a failure does not count. Convergence means *raising K_max further stops mattering in practice.*

### Visualization 6A: Convergence dashboard (2×2)

Four panels showing all the convergence metrics with thresholds. Each panel uses the same x-axis (K_max transitions) so they tell a consistent story.

In [ ]:
# Build a 2x2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Adjacent comparisons sorted by k_max_a
adj_sorted = sorted(summary["adjacent_comparisons"], key=lambda m: m["k_max_a"])
labels = [f"{m['k_max_a']}→{m['k_max_b']}" for m in adj_sorted]

# Panel 1: Spearman rho
ax = axes[0, 0]
rhos = [m["spearman_rho"] for m in adj_sorted]
ax.plot(range(len(rhos)), rhos, "o-", color="steelblue", linewidth=2, markersize=10)
ax.axhline(y=0.95, color="red", linestyle="--", alpha=0.6, label="Threshold (0.95)")
ax.set_xticks(range(len(rhos)))
ax.set_xticklabels(labels)
ax.set_ylim([min(0.7, min(rhos) - 0.05), 1.01])
ax.set_title("Spearman ρ (rank correlation)")
ax.legend()

# Panel 2: Top-50 overlap
ax = axes[0, 1]
top50 = [m.get("top_50_overlap_pct", 0) for m in adj_sorted]
ax.plot(range(len(top50)), top50, "o-", color="coral", linewidth=2, markersize=10)
ax.axhline(y=80, color="red", linestyle="--", alpha=0.6, label="Threshold (80%)")
ax.set_xticks(range(len(top50)))
ax.set_xticklabels(labels)
ax.set_ylim([0, 105])
ax.set_title("Top-50 pair overlap (%)")
ax.legend()

# Panel 3: Mean NN-5 overlap
ax = axes[1, 0]
nn5 = [m.get("mean_nn5_overlap_pct", 0) for m in adj_sorted]
ax.plot(range(len(nn5)), nn5, "o-", color="seagreen", linewidth=2, markersize=10)
ax.set_xticks(range(len(nn5)))
ax.set_xticklabels(labels)
ax.set_ylim([0, 105])
ax.set_title("Mean per-firm NN-5 overlap (%)")

# Panel 4: Effective K ceiling rate
ax = axes[1, 1]
ax.plot(K_MAX_VALUES, k_df["ceiling_rate_pct"], "o-",
        color="mediumpurple", linewidth=2, markersize=10)
ax.set_xticks(K_MAX_VALUES)
ax.set_ylim([0, max(50, k_df["ceiling_rate_pct"].max() * 1.2)])
ax.set_title("Ceiling rate: % of GMM-tier firms with effective K = K_max")

# Verdict annotation
verdict = summary["convergence_verdict"]
converged_at = summary.get("converged_at_kmax")
verdict_str = f"VERDICT: {verdict.upper()}"
if converged_at:
    verdict_str += f" at K_max={converged_at}"
fig.suptitle(verdict_str, fontsize=15, fontweight="bold",
             color="green" if verdict == "converged" else "darkred")

plt.tight_layout()
plt.savefig("03_viz6a_convergence_dashboard.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\n{verdict_str}")
print(f"\nDecision rule: {summary['decision_rule']['definition']}")


### Robust vs model-sensitive pair classification

For Week 3 reporting, we classify candidate pairs into two categories:

- **Robust**: pairs that appear in the top-200 at all K_max settings tested
- **Model-sensitive**: pairs that appear in the top-200 at some but not all K_max settings

Robust pairs are the strongest M&A candidates — their similarity is invariant to specification choices. Model-sensitive pairs should be reported with their K_max profile (e.g., "appears in top-200 at K_max ≥ 20 only").

In [ ]:
# Compute the robust/model-sensitive partition for top-200 pairs
TOP_K = 200
top_200_sets = {}
for k_max in K_MAX_VALUES:
    _, bc_mat = bc_matrices[k_max]
    top_200_sets[k_max] = top_k_pair_set(bc_mat, TOP_K)

# Union of all top-200 pairs across K_max
all_top_pairs = set()
for s in top_200_sets.values():
    all_top_pairs |= s

# Each pair: how many K_max settings does it appear in?
pair_appearances = {p: sum(1 for k in K_MAX_VALUES if p in top_200_sets[k])
                    for p in all_top_pairs}

robust_pairs = {p for p, c in pair_appearances.items() if c == len(K_MAX_VALUES)}
model_sensitive = all_top_pairs - robust_pairs

print(f"Total pairs ever in top-{TOP_K}: {len(all_top_pairs):,}")
print(f"  Robust (in top-{TOP_K} at all K_max): {len(robust_pairs):,} "
      f"({len(robust_pairs)/len(all_top_pairs):.1%})")
print(f"  Model-sensitive: {len(model_sensitive):,} "
      f"({len(model_sensitive)/len(all_top_pairs):.1%})")

# Distribution of appearance counts
counts = pd.Series(list(pair_appearances.values())).value_counts().sort_index()
print(f"\nDistribution of pair appearance counts (out of {len(K_MAX_VALUES)} K_max settings):")
print(counts.to_string())


## Section 7: Implications for M&A Prediction

This is the "so what" section. It connects all the technical findings back to the research question: **can patent portfolios predict M&A pairs in the technology sector?**

### Three findings inform the downstream task:

**1. The distributional representation is fundamentally sound.**

`[POPULATE — based on Section 2 findings]`

The number of technology areas the model identifies scales sensibly with firm size. Larger, more diversified firms have more components; focused firms have fewer. The Bayesian prior produces effective K values that match the economic intuition (logarithmic scaling). The bulk of pairwise similarity rankings are robust to the granularity parameter. **The GMM representation captures genuine technological structure, not noise.**

**2. Top-pair identification requires `[robustness classification | a single converged specification]`.**

`[POPULATE — based on Sections 4 and 6 findings]`

`[If converged]`: The persistent-stability rule identifies K_max=`[K*]` as the production specification. From this point onward, the candidate set of most technologically similar firm pairs is stable across all higher K_max values tested. We adopt K_max=`[K*]` as the production default and keep K_max=`[K*+5]` as a robustness check artifact.

`[If not converged]`: The candidate set is sensitive to K_max within the range tested. We proceed with K_max=`[default]` as the operational specification but explicitly classify all top-200 candidate pairs as robust or model-sensitive. Of the `[N]` pairs that ever appear in the top-200, `[X]` (`[Y]%`) are robust across all K_max settings — these are the strongest M&A candidates. The remaining `[Z]` pairs are reported with their K_max profile. **Sensitivity to K_max is itself a substantive methodological finding** that strengthens, rather than weakens, the credibility of the conclusions we do draw.

**3. The two-tier structure is a feature, not a limitation.**

The 6,304 single-Gaussian firms provide a stable anchor: their distributional representation is parameter-free and their pairwise similarities never change. The 1,645 GMM-tier firms — holding 93% of patents — are where the rich distributional information lives, and where K_max sensitivity is concentrated. This mirrors economic reality: most small firms have a single technology focus, while large firms have complex multi-area portfolios where the boundaries between "areas" are inherently fuzzy.

### Connection to Bena & Li (2014)

Our distributional approach extends the technological overlap variable of Bena and Li (2014) with richer information about the *structure* of overlap — not just "are these firms similar?" but "in which technology areas do they overlap, and how concentrated is each firm's portfolio?" The K_max sensitivity analysis in this notebook provides the evidentiary foundation for treating this measure as a credible input to the M&A prediction model. The trade-off is honest: more information at the cost of one specification choice that requires robustness reporting.

### Recommended Week 3 protocol

`[POPULATE based on verdict — one of the two pre-registered branches from Section 6]`

---

## Notebook Provenance

- **Sweep run ID**: `[populated from convergence_summary.json]`
- **Sweep script**: `scripts/run_kmax_sweep.py` at commit `[git hash]`
- **Decision rule**: pre-registered before sweep launch (see ADR-004 and the executive summary)
- **Data**: `output/week2_inputs/patent_vectors_50d.parquet` from Week 1 production run `20260408T005013Z`
- **Authoritative summary**: `docs/epics/week2_firm_portfolios/kmax_sweep_executive_summary.md`

For methodology details, see:
- `docs/adr/adr_004_k_selection_method.md` — K selection and convergence framework
- `docs/adr/adr_005_minimum_patent_threshold.md` — Three-tier firm classification
- `docs/adr/adr_006_covariance_type.md` — Diagonal covariance and BC formula
- `docs/specs/firm_portfolio_spec.md` — Interface contracts and serialization